In [117]:
!pip install fpdf pymupdf chromadb sentence-transformers openai

# Enterprise RAG Intelligence System
### Multi-Format Ingestion | RBAC Pre-Filtering | Cited Generation
---

## Step 1: Install & Import Dependencies

In [118]:
import os, json, re, math
import pandas as pd
from pathlib import Path
from collections import Counter

# PDF generation & parsing
from fpdf import FPDF
import fitz  # PyMuPDF

# Vector store & embeddings
import chromadb
from chromadb.utils import embedding_functions

In [119]:
# Configuration
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CHROMA_DIR = Path("chroma_db")
ROLES_FILE = DATA_DIR / "User_Roles.json"
LOGS_FILE = DATA_DIR / "System_Logs.json"
CSV_FILE = DATA_DIR / "Financial_Records.csv"
HR_TEXT_FILE = DATA_DIR / "HR_Policy_Content.txt"
HR_PDF_FILE = DATA_DIR / "HR_Policy.pdf"

In [120]:
import json
import pandas as pd

# 1. User Roles
roles_data = {
    "Alice": {"roles": ["HR", "Admin"], "clearance_level": 3},
    "Bob": {"roles": ["Finance"], "clearance_level": 2},
    "Charlie": {"roles": ["General_Employee"], "clearance_level": 1},
    "Diana": {"roles": ["HR", "Finance", "Admin"], "clearance_level": 3}
}
with open(ROLES_FILE, 'w') as f:
    json.dump(roles_data, f)

# 2. Financial Records
financial_data = [
    {"Quarter": "Q3", "Year": 2025, "Department": "Sales", "Region": "North", "Revenue": 500000, "Expenses": 300000, "Net_Profit": 200000, "Growth_Rate": 5},
    {"Quarter": "Q3", "Year": 2025, "Department": "Engineering", "Region": "West", "Revenue": 0, "Expenses": 450000, "Net_Profit": -450000, "Growth_Rate": 2}
]
pd.DataFrame(financial_data).to_csv(CSV_FILE, index=False)

# 3. System Logs
logs_data = [
    {"timestamp": "2025-05-10 09:00:00", "severity": "ERROR", "service": "Payroll", "error_code": "PY-404", "message": "Database connection timeout", "allowed_roles": ["Admin"]},
    {"timestamp": "2025-05-10 10:30:00", "severity": "INFO", "service": "Gateway", "error_code": "GW-200", "message": "Service heartbeat stable", "allowed_roles": ["Admin", "Finance"]}
]
with open(LOGS_FILE, 'w') as f:
    json.dump(logs_data, f)

# 4. HR Policy Text Source
hr_text = """SECTION 1: OVERVIEW\nACME CORPORATION values its employees.\n\nSECTION 2: PTO\nSenior employees receive 25 days of PTO per year.\n\nSECTION 3: PARENTAL LEAVE\nParents receive 12 weeks of paid leave after 1 year of service."""
HR_TEXT_FILE.write_text(hr_text)

print("[OK] Synthetic data files created.")

[OK] Synthetic data files created.


## Step 2: Generate Synthetic HR_Policy.pdf

In [121]:
def generate_hr_pdf(text_path: Path, pdf_path: Path):
    """Convert HR policy text file into a proper PDF document."""
    content = text_path.read_text(encoding="utf-8")
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, "ACME CORPORATION", ln=True, align="C")
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 6, "Human Resources Policy Manual v3.2", ln=True, align="C")
    pdf.ln(8)
    pdf.set_font("Helvetica", "", 9)
    for line in content.split("\n"):
        line = line.strip()
        if not line:
            pdf.ln(3)
        elif line.startswith("SECTION") or line.startswith("ACME"):
            pdf.set_font("Helvetica", "B", 12)
            safe = line.encode("latin-1", "replace").decode("latin-1")
            pdf.cell(0, 7, safe, ln=True)
            pdf.set_font("Helvetica", "", 9)
        elif line and line[0].isdigit() and "." in line[:4]:
            pdf.set_font("Helvetica", "B", 10)
            safe = line.encode("latin-1", "replace").decode("latin-1")
            pdf.cell(0, 6, safe, ln=True)
            pdf.set_font("Helvetica", "", 9)
        else:
            safe = line.encode("latin-1", "replace").decode("latin-1")
            pdf.multi_cell(0, 5, safe)
    pdf.output(str(pdf_path))
    print(f"[OK] Generated PDF: {pdf_path} ({pdf_path.stat().st_size:,} bytes)")

generate_hr_pdf(HR_TEXT_FILE, HR_PDF_FILE)

[OK] Generated PDF: data/HR_Policy.pdf (1,331 bytes)


## Step 3: Load User Roles (RBAC Mapping)

In [122]:
with open(ROLES_FILE, "r") as f:
    USER_ROLES_DB = json.load(f)

def get_user_roles(username: str) -> list:
    """Look up a user's roles from the RBAC database."""
    entry = USER_ROLES_DB.get(username)
    if entry is None:
        raise ValueError(f"Unknown user: {username}")
    return entry["roles"]

# Display user-role matrix
print("=" * 55)
print(f"{'User':<12} {'Roles':<35} {'Clearance'}")
print("=" * 55)
for user, info in USER_ROLES_DB.items():
    print(f"{user:<12} {', '.join(info['roles']):<35} {info['clearance_level']}")

User         Roles                               Clearance
Alice        HR, Admin                           3
Bob          Finance                             2
Charlie      General_Employee                    1
Diana        HR, Finance, Admin                  3


## Step 4: Ingest & Tag All Data Sources
Every chunk gets `allowed_roles` metadata for RBAC pre-filtering.

In [123]:
class DocumentChunk:
    def __init__(self, text, source, chunk_type, allowed_roles, extra_meta=None):
        self.text = text
        self.source = source
        self.chunk_type = chunk_type
        self.allowed_roles = allowed_roles
        self.extra_meta = extra_meta or {}
    def to_metadata(self):
        meta = {'source': self.source, 'chunk_type': self.chunk_type, 'allowed_roles_str': ','.join(sorted(self.allowed_roles))}
        meta.update(self.extra_meta)
        return meta

In [124]:
def ingest_pdf(pdf_path: Path) -> list:
    doc = fitz.open(str(pdf_path))
    chunks = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        sections = re.split(r'(SECTION \d+:.*)', text)
        for section in sections:
            section = section.strip()
            if len(section) < 20: continue
            chunks.append(DocumentChunk(text=section, source=f"HR_Policy.pdf (Page {page_num + 1})", chunk_type="policy_document", allowed_roles=["HR", "Admin"], extra_meta={"page": str(page_num + 1)}))
    doc.close()
    print(f"[PDF] Ingested {len(chunks)} chunks")
    return chunks

In [125]:
def ingest_csv(csv_path: Path) -> list:
    df = pd.read_csv(csv_path)
    chunks = []
    for _, row in df.iterrows():
        text = f"{row['Quarter']} {row['Year']} | {row['Department']}: Revenue=${row['Revenue']}"
        chunks.append(DocumentChunk(text=text, source="Financial_Records.csv", chunk_type="financial_data", allowed_roles=["Finance", "Admin"]))
    chunks.append(DocumentChunk(text=f"Total Revenue: {df['Revenue'].sum()}", source="Financial_Summary", chunk_type="financial_summary", allowed_roles=["Finance", "Admin", "General_Employee"]))
    print(f"[CSV] Ingested {len(chunks)} chunks")
    return chunks

In [126]:
def ingest_json_logs(json_path: Path) -> list:
    with open(json_path, 'r') as f: logs = json.load(f)
    chunks = [DocumentChunk(text=f"{l['severity']}: {l['message']}", source="System_Logs.json", chunk_type="system_log", allowed_roles=l['allowed_roles']) for l in logs]
    print(f"[JSON] Ingested {len(chunks)} chunks")
    return chunks

In [127]:
all_chunks = []
all_chunks.extend(ingest_pdf(HR_PDF_FILE))
all_chunks.extend(ingest_csv(CSV_FILE))
all_chunks.extend(ingest_json_logs(LOGS_FILE))
print(f"Total chunks ingested: {len(all_chunks)}")

[PDF] Ingested 5 chunks
[CSV] Ingested 3 chunks
[JSON] Ingested 2 chunks
Total chunks ingested: 10


## Step 5: Build ChromaDB Vector Store

In [128]:
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.get_or_create_collection(name="enterprise_rag", embedding_function=embed_fn)
collection.add(ids=[f"id_{i}" for i in range(len(all_chunks))], documents=[c.text for c in all_chunks], metadatas=[c.to_metadata() for c in all_chunks])
print("Vector store updated.")

Vector store updated.


In [129]:
# Debug: Inspect stored metadata to verify RBAC filtering format
import json
debug_results = collection.get(limit=5, include=['metadatas'])
print("Sample Metadata in Vector Store:")
print(json.dumps(debug_results['metadatas'], indent=2))

Sample Metadata in Vector Store:
[
  {
    "allowed_roles_str": "Admin,HR",
    "chunk_type": "policy_document",
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1"
  },
  {
    "allowed_roles_str": "Admin,HR",
    "chunk_type": "policy_document",
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1"
  },
  {
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1",
    "chunk_type": "policy_document",
    "allowed_roles_str": "Admin,HR"
  },
  {
    "source": "HR_Policy.pdf (Page 1)",
    "allowed_roles_str": "Admin,HR",
    "page": "1",
    "chunk_type": "policy_document"
  },
  {
    "chunk_type": "policy_document",
    "page": "1",
    "allowed_roles_str": "Admin,HR",
    "source": "HR_Policy.pdf (Page 1)"
  }
]


In [130]:
# Debug: Inspect stored metadata to verify RBAC filtering format
debug_results = collection.get(limit=5, include=['metadatas', 'documents'])
import json
print(json.dumps(debug_results['metadatas'], indent=2))

[
  {
    "allowed_roles_str": "Admin,HR",
    "source": "HR_Policy.pdf (Page 1)",
    "chunk_type": "policy_document",
    "page": "1"
  },
  {
    "allowed_roles_str": "Admin,HR",
    "chunk_type": "policy_document",
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1"
  },
  {
    "chunk_type": "policy_document",
    "allowed_roles_str": "Admin,HR",
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1"
  },
  {
    "chunk_type": "policy_document",
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1",
    "allowed_roles_str": "Admin,HR"
  },
  {
    "source": "HR_Policy.pdf (Page 1)",
    "page": "1",
    "allowed_roles_str": "Admin,HR",
    "chunk_type": "policy_document"
  }
]


## Step 6: RBAC-Filtered Retrieval

In [131]:
def rbac_retrieve(query: str, username: str, top_k: int = 5) -> dict:
    user_roles = set(get_user_roles(username))
    # Fetch more results initially to account for filtering
    results = collection.query(query_texts=[query], n_results=top_k * 3, include=["documents", "metadatas", "distances"])

    filtered_docs = []
    filtered_metas = []
    filtered_distances = []

    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        # Split the stored string back into a set of roles
        allowed = set(meta.get("allowed_roles_str", "").split(","))
        if user_roles.intersection(allowed):
            filtered_docs.append(doc)
            filtered_metas.append(meta)
            filtered_distances.append(dist)

        if len(filtered_docs) >= top_k:
            break

    return {
        "query": query,
        "username": username,
        "roles": list(user_roles),
        "documents": filtered_docs,
        "metadatas": filtered_metas,
        "distances": filtered_distances
    }

## Step 7: Citation-Aware Prompt Construction

In [132]:
def build_cited_prompt(query: str, retrieved: dict) -> str:
    context_blocks = [f"Context ID [{i}] (Source: {m['source']}):\n{d}" for i, (d, m) in enumerate(zip(retrieved["documents"], retrieved["metadatas"]), 1)]
    return f"Answer using ONLY this context:\n\n{'\n\n'.join(context_blocks)}\n\nQuestion: {query}\nAnswer:"

## Step 8: Confidence Scoring

In [133]:
def compute_confidence(retrieved: dict) -> dict:
    sims = [1 - d for d in retrieved["distances"]]
    avg_sim = sum(sims) / len(sims) if sims else 0
    return {"avg_similarity": avg_sim, "num_results": len(sims), "confidence_tier": "HIGH" if avg_sim > 0.5 else "MEDIUM"}

In [134]:
from pydantic import BaseModel, Field
from typing import List

class RAGResponse(BaseModel):
    """Schema for the final validated RAG answer."""
    summary: str = Field(description="A concise answer derived only from the provided context.")
    citations: List[int] = Field(description="List of Context IDs used to generate the answer.")
    information_available: bool = Field(description="Whether the requested info was found and authorized.")
    confidence_score: float = Field(description="The similarity score between query and context.")

## Step 9: LLM Generation (with Offline Fallback)

In [135]:
import os

def generate_answer_offline(prompt: str, retrieved: dict) -> str:
    """Rule-based fallback when no LLM API key is available."""
    docs = retrieved["documents"]
    metas = retrieved["metadatas"]
    query_lower = retrieved["query"].lower()
    answer_parts = ["Based on the retrieved documents:\n"]
    for i, (doc, meta) in enumerate(zip(docs, metas), 1):
        doc_lower = doc.lower()
        query_words = set(query_lower.split())
        overlap = sum(1 for w in query_words if w in doc_lower)
        if overlap >= 1:
            sentences = [s.strip() for s in doc.split(".") if len(s.strip()) > 20]
            if sentences:
                best = max(sentences, key=lambda s: sum(1 for w in query_words if w in s.lower()))
                answer_parts.append(f"- {best.strip('.')}. [{i}]")
    if len(answer_parts) == 1:
        answer_parts.append("Information not accessible or unavailable for your role.")
    answer_parts.append(f"\n\nSources: " + ", ".join(f"[{i+1}] {m['source']}" for i, m in enumerate(metas)))
    return "\n".join(answer_parts)

def generate_answer(prompt: str, retrieved: dict) -> str:
    """Use Groq API for generation, falling back to offline if needed."""
    # Using the Groq key provided by the user
    groq_api_key = "your-groq-api-key-here"

    if groq_api_key:
        try:
            from openai import OpenAI
            # Initialize client for Groq API using OpenAI-compatible interface
            client = OpenAI(
                base_url="https://api.groq.com/openai/v1",
                api_key=groq_api_key
            )
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=1024
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"[Groq LLM] API call failed: {e}. Using offline fallback.")

    print("[LLM] No valid API key found. Using offline rule-based generation.")
    return generate_answer_offline(prompt, retrieved)

In [ ]:
def generate_structured_answer(prompt: str, retrieved: dict) -> RAGResponse:
    """Use Groq + Pydantic for structured, validated generation."""
    groq_api_key = "enter-api-key"
    confidence = compute_confidence(retrieved)

    if not retrieved["documents"]:
        return RAGResponse(
            summary="No documents retrieved for your role/query.",
            citations=[],
            information_available=False,
            confidence_score=0.0
        )

    try:
        from openai import OpenAI
        client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)

        # Instructing LLM to return JSON matching our Pydantic schema
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Return ONLY valid JSON."},
                {"role": "user", "content": f"{prompt}\n\nReturn JSON with fields: summary, citations (list of ints), information_available (bool)."}
            ],
            response_format={ "type": "json_object" },
            temperature=0.1
        )

        raw_json = json.loads(response.choices[0].message.content)
        # Validate with Pydantic
        return RAGResponse(
            summary=raw_json.get("summary", ""),
            citations=raw_json.get("citations", []),
            information_available=raw_json.get("information_available", True),
            confidence_score=confidence["avg_similarity"]
        )
    except Exception as e:
        print(f"Structure Generation Error: {e}")
        return RAGResponse(summary="Error generating structured response.", citations=[], information_available=False, confidence_score=0.0)

## Step 10: Complete Query Pipeline

In [137]:
def query_with_rbac(username: str, question: str, top_k: int = 5, verbose: bool = True) -> RAGResponse:
    """Full RAG pipeline returning structured, validated output."""
    if verbose:
        print(f"\n{'='*70}")
        print(f"  USER: {username} | ROLES: {get_user_roles(username)}")
        print(f"  QUERY: {question}")
        print(f"{'='*70}")

    # 1. Retrieval with RBAC
    retrieved = rbac_retrieve(question, username, top_k)

    # 2. Prompt Construction
    prompt = build_cited_prompt(question, retrieved)

    # 3. Structured Generation (using Pydantic + Groq)
    structured_res = generate_structured_answer(prompt, retrieved)

    if verbose:
        print(f"\n  Confidence: {structured_res.confidence_score:.3f}")
        print(f"  Information Available: {structured_res.information_available}")
        print(f"\n  SUMMARY:\n  {'-'*60}")
        print(f"  {structured_res.summary}")
        print(f"  Citations: {structured_res.citations}")
        print(f"  {'-'*60}")

    return structured_res

In [138]:
# Final Validation: Structured RBAC Responses
print("--- STARTING STRUCTURED VALIDATION ---\n")

# 1. Alice (HR/Admin) - Should get full details + citations
response_alice = query_with_rbac("Alice", "What is the policy for parental leave?")

# 2. Bob (Finance) - Should get financial info but be restricted from HR details
response_bob_finance = query_with_rbac("Bob", "What was the revenue for Engineering?")
response_bob_hr = query_with_rbac("Bob", "What is the PTO policy?")

# 3. Charlie (Employee) - Restricted from sensitive data
response_charlie = query_with_rbac("Charlie", "Show me the payroll error logs")

print("\n--- VALIDATION COMPLETE ---")

--- STARTING STRUCTURED VALIDATION ---


  USER: Alice | ROLES: ['HR', 'Admin']
  QUERY: What is the policy for parental leave?

  Confidence: 0.397
  Information Available: True

  SUMMARY:
  ------------------------------------------------------------
  Parents receive 12 weeks of paid leave after 1 year of service
  Citations: [1, 2]
  ------------------------------------------------------------

  USER: Bob | ROLES: ['Finance']
  QUERY: What was the revenue for Engineering?

  Confidence: 0.371
  Information Available: True

  SUMMARY:
  ------------------------------------------------------------
  The revenue for Engineering was $0
  Citations: [1]
  ------------------------------------------------------------

  USER: Bob | ROLES: ['Finance']
  QUERY: What is the PTO policy?

  Confidence: 0.105
  Information Available: False

  SUMMARY:
  ------------------------------------------------------------
  No information available about PTO policy
  Citations: []
  ------------------

## Demo: RBAC Access Control in Action

### Demo 1: Alice (HR + Admin) — Can access HR policies

In [139]:
result_alice = query_with_rbac("Alice", "How many PTO days do senior employees get per year?")


  USER: Alice | ROLES: ['HR', 'Admin']
  QUERY: How many PTO days do senior employees get per year?

  Confidence: 0.406
  Information Available: True

  SUMMARY:
  ------------------------------------------------------------
  Senior employees receive 25 days of PTO per year
  Citations: [1]
  ------------------------------------------------------------


### Demo 2: Bob (Finance) — Can access financial data

In [140]:
result_bob = query_with_rbac("Bob", "What was the total revenue for Sales in Q3 2025?")


  USER: Bob | ROLES: ['Finance']
  QUERY: What was the total revenue for Sales in Q3 2025?

  Confidence: 0.569
  Information Available: True

  SUMMARY:
  ------------------------------------------------------------
  The total revenue for Sales in Q3 2025 was $500000
  Citations: [1, 3]
  ------------------------------------------------------------


### Demo 3: Charlie (General Employee) — Restricted access

In [141]:
result_charlie = query_with_rbac("Charlie", "Show me the payroll error logs")


  USER: Charlie | ROLES: ['General_Employee']
  QUERY: Show me the payroll error logs

  Confidence: 0.201
  Information Available: False

  SUMMARY:
  ------------------------------------------------------------
  No payroll error logs available in the given context
  Citations: []
  ------------------------------------------------------------


### Demo 4: Diana (HR + Finance + Admin) — Full access

In [142]:
result_diana = query_with_rbac("Diana", "Summarize the latest system errors and HR policy on parental leave")


  USER: Diana | ROLES: ['HR', 'Finance', 'Admin']
  QUERY: Summarize the latest system errors and HR policy on parental leave

  Confidence: 0.388
  Information Available: False

  SUMMARY:
  ------------------------------------------------------------
  No system errors reported, parental leave policy provides 12 weeks of paid leave after 1 year of service
  Citations: [1, 2]
  ------------------------------------------------------------


### Demo 5: RBAC Security Verification
Bob (Finance only) tries to access HR data — should be blocked.

In [143]:
result_bob_hr = query_with_rbac("Bob", "What is the parental leave policy?")


  USER: Bob | ROLES: ['Finance']
  QUERY: What is the parental leave policy?

  Confidence: -0.020
  Information Available: False

  SUMMARY:
  ------------------------------------------------------------
  No information available on parental leave policy
  Citations: []
  ------------------------------------------------------------


## Summary & Confidence Matrix

In [144]:
print("Executing full demo suite with Groq LLM (Llama 3.3-70B)...\n")

# Re-run all personas to show the difference with a real LLM
query_with_rbac("Alice", "How many PTO days do senior employees get per year?")
query_with_rbac("Bob", "What was the total revenue for Sales in Q3 2025?")
query_with_rbac("Charlie", "Show me the payroll error logs")
query_with_rbac("Diana", "Summarize the latest system errors and HR policy on parental leave")

Executing full demo suite with Groq LLM (Llama 3.3-70B)...


  USER: Alice | ROLES: ['HR', 'Admin']
  QUERY: How many PTO days do senior employees get per year?

  Confidence: 0.406
  Information Available: True

  SUMMARY:
  ------------------------------------------------------------
  Senior employees receive 25 days of PTO per year
  Citations: [1]
  ------------------------------------------------------------

  USER: Bob | ROLES: ['Finance']
  QUERY: What was the total revenue for Sales in Q3 2025?

  Confidence: 0.569
  Information Available: True

  SUMMARY:
  ------------------------------------------------------------
  The total revenue for Sales in Q3 2025 was $500000
  Citations: [1, 3]
  ------------------------------------------------------------

  USER: Charlie | ROLES: ['General_Employee']
  QUERY: Show me the payroll error logs

  Confidence: 0.201
  Information Available: False

  SUMMARY:
  ------------------------------------------------------------
  No payroll e

RAGResponse(summary='No system errors reported, parental leave policy provides 12 weeks of paid leave after 1 year of service', citations=[1, 2], information_available=False, confidence_score=0.38827338218688967)

## Project Credits & Contact

Developed by **Abhay Gupta**

*   **GitHub:** [Abs6187](https://github.com/Abs6187)
*   **LinkedIn:** [Abhay Gupta](https://www.linkedin.com/in/abhay-gupta-197b17264/)
*   **X (Twitter):** [@AbhayGu19265651](https://x.com/AbhayGu19265651)

---
*© 2026 Abhay Gupta. All Rights Reserved.*